# Task 3 – Case Studies with PDA and CFG
**Formal Language and Automata (21CSC301T)**

This notebook implements the three case studies in the assignment:
1. Balanced parentheses using a Pushdown Automaton (PDA)
2. Arithmetic expression validation using a Context-Free Grammar (CFG) and stack-based parsing
3. Centre-marked palindrome checking for the language `w c w^R`

All examples are included in the notebook. No external dataset is required.

In [ ]:
# Cell 1: Environment setup
import sys, subprocess, importlib.util

if importlib.util.find_spec("pandas") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas"])

import pandas as pd
from IPython.display import display, Markdown

print("Environment ready.")

## 1. Why a stack is needed

A finite automaton has only a fixed number of states. A PDA adds a stack, which provides last-in-first-out (LIFO) memory. This makes it possible to remember an arbitrary number of opening parentheses or symbols that must later be matched.

In [ ]:
# Cell 2: PDA for balanced parentheses
# Z0 is the bottom-of-stack marker. Each '(' pushes X.
# Each ')' pops X. Accept when input ends and only Z0 remains.

def balanced_parentheses_pda(text, show_trace=False):
    stack = ["Z0"]
    trace = []

    for index, symbol in enumerate(text, start=1):
        if symbol == "(":
            stack.append("X")
            action = "Push X"
        elif symbol == ")":
            if stack[-1] == "Z0":
                trace.append({"Step": index, "Symbol": symbol,
                              "Action": "No matching opening bracket",
                              "Stack": stack.copy()})
                return False, trace
            stack.pop()
            action = "Pop X"
        else:
            trace.append({"Step": index, "Symbol": symbol,
                          "Action": "Invalid symbol", "Stack": stack.copy()})
            return False, trace

        trace.append({"Step": index, "Symbol": symbol,
                      "Action": action, "Stack": stack.copy()})

    accepted = stack == ["Z0"]
    if accepted:
        trace.append({"Step": len(text)+1, "Symbol": "ε",
                      "Action": "Accept: only Z0 remains", "Stack": stack.copy()})
    else:
        trace.append({"Step": len(text)+1, "Symbol": "END",
                      "Action": "Reject: unmatched opening bracket", "Stack": stack.copy()})
    return accepted, trace


parentheses_tests = [
    ("()", True), ("(())", True), ("()()", True), ("((()()))", True),
    ("(()())", True), ("(", False), (")", False), ("))(", False),
    ("((())", False), (")(", False)
]

paren_results = []
for value, expected in parentheses_tests:
    actual, _ = balanced_parentheses_pda(value)
    paren_results.append({
        "Input": value,
        "Result": "ACCEPTED" if actual else "REJECTED",
        "Expected": "ACCEPTED" if expected else "REJECTED",
        "Test": "PASS" if actual == expected else "FAIL"
    })

display(pd.DataFrame(paren_results))

print("\nDetailed stack trace for input (())")
accepted, trace = balanced_parentheses_pda("(())", show_trace=True)
display(pd.DataFrame([
    {"Step": r["Step"], "Symbol": r["Symbol"], "Action": r["Action"],
     "Stack": str(r["Stack"])} for r in trace
]))
print("Final result:", "ACCEPTED" if accepted else "REJECTED")

## 2. Arithmetic syntax validation using CFG

The grammar used in the assignment is:

- `E → E + T | T`
- `T → T * F | F`
- `F → (E) | id`

`E` handles addition, `T` handles multiplication, and `F` handles identifiers or parenthesized expressions. This structure gives multiplication higher precedence than addition.

The parser below uses a stack of grammar symbols and input tokens. It accepts only expressions that can be derived from this grammar.

In [ ]:
# Cell 3: Tokenizer and stack-based predictive parser for the equivalent CFG
# The grammar is rewritten without left recursion:
# E  -> T E'
# E' -> + T E' | epsilon
# T  -> F T'
# T' -> * F T' | epsilon
# F  -> ( E ) | id

import re

TOKEN_PATTERN = re.compile(r"\s*(id|[()+*])")

def tokenize_expression(expression):
    tokens = []
    pos = 0
    while pos < len(expression):
        match = TOKEN_PATTERN.match(expression, pos)
        if not match:
            raise ValueError(f"Invalid token near: {expression[pos:]}")
        tokens.append(match.group(1))
        pos = match.end()
    tokens.append("$")
    return tokens


def parse_expression(expression, return_trace=False):
    try:
        tokens = tokenize_expression(expression)
    except ValueError:
        return (False, []) if return_trace else False

    # Stack top is at the end of the list.
    stack = ["$", "E"]
    index = 0
    trace = []
    productions = {
        "E": ["T", "E'"],
        "E'": ["+", "T", "E'"],  # selected only when lookahead is +
        "T": ["F", "T'"],
        "T'": ["*", "F", "T'"],  # selected only when lookahead is *
        "F": ["(", "E", ")"]     # or id
    }

    max_steps = 500
    steps = 0
    while stack and steps < max_steps:
        steps += 1
        top = stack[-1]
        lookahead = tokens[index]

        if top == "$" and lookahead == "$":
            trace.append({"Step": steps, "Top": top, "Input": lookahead,
                          "Action": "Match end marker", "Stack": stack.copy()})
            stack.pop()
            return (True, trace) if return_trace else True

        if top in ("E", "E'", "T", "T'", "F"):
            stack.pop()
            if top == "E":
                replacement = ["E'", "T"]
            elif top == "E'":
                replacement = ["E'", "T", "+"] if lookahead == "+" else []
            elif top == "T":
                replacement = ["T'", "F"]
            elif top == "T'":
                replacement = ["T'", "F", "*"] if lookahead == "*" else []
            else:  # F
                if lookahead == "id":
                    replacement = ["id"]
                elif lookahead == "(":
                    replacement = [")", "E", "("]
                else:
                    return (False, trace) if return_trace else False

            stack.extend(replacement)
            action = "Expand " + top + (" -> " + " ".join(reversed(replacement))
                                        if replacement else " -> ε")
            trace.append({"Step": steps, "Top": top, "Input": lookahead,
                          "Action": action, "Stack": stack.copy()})
        else:
            if top == lookahead:
                stack.pop()
                index += 1
                trace.append({"Step": steps, "Top": top, "Input": lookahead,
                              "Action": "Match token", "Stack": stack.copy()})
            else:
                return (False, trace) if return_trace else False

    result = False
    return (result, trace) if return_trace else result


arithmetic_tests = [
    ("id", True), ("id+id", True), ("id*id+id", True),
    ("(id+id)*id", True), ("id*(id+id*id)", True),
    ("id+", False), ("+id", False), ("id*+id", False),
    ("(id+id", False), ("id id", False)
]

arith_results = []
for value, expected in arithmetic_tests:
    actual = parse_expression(value)
    arith_results.append({
        "Input": value,
        "Result": "ACCEPTED" if actual else "REJECTED",
        "Expected": "ACCEPTED" if expected else "REJECTED",
        "Test": "PASS" if actual == expected else "FAIL"
    })

display(pd.DataFrame(arith_results))

example = "(id+id)*id"
accepted, arithmetic_trace = parse_expression(example, return_trace=True)
print(f"\nParsing trace for {example}")
display(pd.DataFrame([
    {"Step": r["Step"], "Top": r["Top"], "Input": r["Input"],
     "Action": r["Action"], "Stack": str(r["Stack"])}
    for r in arithmetic_trace
]))
print("Final result:", "ACCEPTED" if accepted else "REJECTED")

## 3. Centre-marked palindrome PDA (`w c w^R`)

The input contains a centre marker `c`. In the first phase, the PDA pushes `A` for every `a` and `B` for every `b`. On reading `c`, it switches to the matching phase. It then pops the matching symbol for each character on the right side. It accepts when the input is exhausted and only `Z0` remains.

In [ ]:
# Cell 4: Centre-marked palindrome PDA
def palindrome_pda(text, return_trace=False):
    stack = ["Z0"]
    phase = "push"
    trace = []
    centre_count = 0

    for step, ch in enumerate(text, start=1):
        if phase == "push":
            if ch == "a":
                stack.append("A")
                action = "Push A"
            elif ch == "b":
                stack.append("B")
                action = "Push B"
            elif ch == "c":
                phase = "match"
                centre_count += 1
                action = "Read centre c; switch to matching phase"
            else:
                return (False, trace) if return_trace else False
        else:
            if ch == "a" and stack[-1] == "A":
                stack.pop()
                action = "Pop A"
            elif ch == "b" and stack[-1] == "B":
                stack.pop()
                action = "Pop B"
            else:
                return (False, trace) if return_trace else False

        trace.append({"Step": step, "Symbol": ch, "Phase": phase,
                      "Action": action, "Stack": stack.copy()})

    accepted = centre_count == 1 and phase == "match" and stack == ["Z0"]
    trace.append({"Step": len(text)+1, "Symbol": "END", "Phase": phase,
                  "Action": "Accept" if accepted else "Reject",
                  "Stack": stack.copy()})
    return (accepted, trace) if return_trace else accepted


palindrome_tests = [
    ("c", True), ("aca", True), ("bcb", True), ("abbcbba", True),
    ("baacaab", True), ("acb", False), ("abcca", False),
    ("abbcbb", False), ("abc", False), ("aabaa", False)
]

pal_results = []
for value, expected in palindrome_tests:
    actual = palindrome_pda(value)
    pal_results.append({
        "Input": value,
        "Result": "ACCEPTED" if actual else "REJECTED",
        "Expected": "ACCEPTED" if expected else "REJECTED",
        "Test": "PASS" if actual == expected else "FAIL"
    })

display(pd.DataFrame(pal_results))

example = "abbcbba"
accepted, pal_trace = palindrome_pda(example, return_trace=True)
print(f"\nStack trace for {example}")
display(pd.DataFrame([
    {"Step": r["Step"], "Symbol": r["Symbol"], "Phase": r["Phase"],
     "Action": r["Action"], "Stack": str(r["Stack"])}
    for r in pal_trace
]))
print("Final result:", "ACCEPTED" if accepted else "REJECTED")

## 4. Master verification – all three case studies

This cell executes all 30 test strings (10 per case study) and displays the overall result.

In [ ]:
# Cell 5: Combined test runner
master_results = []

for row in paren_results:
    master_results.append({"Case Study": "Balanced Parentheses", **row})
for row in arith_results:
    master_results.append({"Case Study": "Arithmetic CFG", **row})
for row in pal_results:
    master_results.append({"Case Study": "Palindrome PDA", **row})

master_df = pd.DataFrame(master_results)
display(master_df)

total = len(master_df)
passed = int((master_df["Test"] == "PASS").sum())
failed = total - passed

summary = pd.DataFrame([
    {"Case Study": "Balanced Parentheses", "Tests": len(paren_results),
     "Passed": sum(r["Test"] == "PASS" for r in paren_results)},
    {"Case Study": "Arithmetic CFG", "Tests": len(arith_results),
     "Passed": sum(r["Test"] == "PASS" for r in arith_results)},
    {"Case Study": "Palindrome PDA", "Tests": len(pal_results),
     "Passed": sum(r["Test"] == "PASS" for r in pal_results)}
])
display(summary)

print("=" * 52)
print(f"TOTAL TEST STRINGS: {total}")
print(f"PASSED: {passed}")
print(f"FAILED: {failed}")
print("Verification status:", "ALL TESTS PASSED" if failed == 0 else "SOME TESTS FAILED")

In [ ]:
# Cell 6: Simple stack concept illustration
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

def draw_stack(ax, symbols, title):
    ax.set_title(title, weight="bold")
    ax.set_xlim(0, 2)
    ax.set_ylim(0, max(4, len(symbols)+1))
    ax.axis("off")
    for i, symbol in enumerate(symbols):
        y = i
        rect = Rectangle((0.55, y), 0.9, 0.65, fill=False, linewidth=1.5)
        ax.add_patch(rect)
        ax.text(1.0, y+0.32, symbol, ha="center", va="center")
    ax.text(1.0, len(symbols)+0.15, "TOP", ha="center", fontsize=9)

draw_stack(axes[0], ["Z0", "X", "X"], "Parentheses stack after '(('")
draw_stack(axes[1], ["Z0", "A", "B"], "Palindrome stack after 'ab'")
plt.tight_layout()
plt.show()

## 5. Conclusion

The three case studies demonstrate how a PDA/stack or CFG parser handles structures that need more than finite-state memory:
- Balanced parentheses: the stack counts nested opening brackets and matches closing brackets.
- Arithmetic expressions: the CFG describes valid expression structure and precedence.
- Centre-marked palindrome: the stack stores the first half and matches it in reverse after `c`.

The notebook reports the result of each test and a combined verification summary.